In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!pip -q install -U transformers datasets soundfile accelerate jiwer

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [3]:
%%writefile train.py
import argparse, io, os, time, itertools, torch, soundfile as sf
from dataclasses import dataclass
from datasets import load_dataset, Audio
from huggingface_hub import login, HfApi
from transformers import (AutoProcessor, MoonshineForConditionalGeneration,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)

ap = argparse.ArgumentParser()
ap.add_argument("--steps", type=int, default=4000)
ap.add_argument("--smoke", type=int, default=0)     # >0: stream this many clips, no full download
ap.add_argument("--push", type=int, default=1)
ap.add_argument("--lr", type=float, default=1e-4)
ap.add_argument("--bs", type=int, default=16)
ap.add_argument("--accum", type=int, default=4)
ap.add_argument("--fp32", type=int, default=0)
a = ap.parse_args()

HF = os.environ["HF_TOKEN"]; login(HF)
DATA, OUT, BASE = "PrathamR777/hinglish-moonshine", "PrathamR777/moonshine-tiny-hinglish", "UsefulSensors/moonshine-tiny"
if a.push:
    HfApi(token=HF).create_repo(OUT, private=True, exist_ok=True)   # fail early if token can't write
cap = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
bf16 = cap >= 8 and not a.fp32
fp16 = 0 < cap < 8 and not a.fp32
print("cuda:", torch.cuda.is_available(), "| capability:", cap, "| bf16:", bf16, "| fp16:", fp16)

proc = AutoProcessor.from_pretrained(BASE)
model = MoonshineForConditionalGeneration.from_pretrained(BASE)
if a.smoke:
    s = load_dataset(DATA, split="train", streaming=True, token=HF).cast_column("audio", Audio(decode=False))
    train_ds = list(itertools.islice(iter(s), a.smoke))
else:
    train_ds = load_dataset(DATA, split="train", token=HF).cast_column("audio", Audio(decode=False))
print("train clips:", len(train_ds), "| params (M):", sum(p.numel() for p in model.parameters()) / 1e6)

@dataclass
class Collator:
    proc: object
    def __call__(self, batch):
        wavs = [sf.read(io.BytesIO(b["audio"]["bytes"]), dtype="float32")[0] for b in batch]
        f = self.proc.feature_extractor(wavs, sampling_rate=16000, return_tensors="pt",
                                        padding=True, return_attention_mask=True)
        tk = self.proc.tokenizer
        ids = [tk(b["text"], add_special_tokens=False).input_ids + [tk.eos_token_id] for b in batch]
        L = max(map(len, ids))
        labels = torch.full((len(ids), L), -100, dtype=torch.long)
        for i, x in enumerate(ids):
            labels[i, :len(x)] = torch.tensor(x)
        out = {"input_values": f["input_values"], "labels": labels}
        if "attention_mask" in f:
            out["attention_mask"] = f["attention_mask"]
        return out

args = Seq2SeqTrainingArguments(
    output_dir="out", per_device_train_batch_size=a.bs, gradient_accumulation_steps=a.accum,
    learning_rate=a.lr, lr_scheduler_type="cosine", warmup_steps=max(3, min(300, a.steps // 10)),
    max_steps=a.steps, bf16=bf16, fp16=fp16, save_steps=500, save_total_limit=2,
    logging_steps=10, remove_unused_columns=False, dataloader_num_workers=2, report_to="none")
t0 = time.time()
Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, data_collator=Collator(proc)).train()
print(f"training took {time.time()-t0:.0f}s for {a.steps} steps = {(time.time()-t0)/a.steps:.2f} s/step")
model.save_pretrained("final"); proc.save_pretrained("final")
if a.push:
    model.push_to_hub(OUT, private=True); proc.push_to_hub(OUT, private=True)
    print("pushed", OUT)

Writing train.py


In [4]:
s = open("train.py").read()
s = s.replace("    proc: object\n", "    proc: object\n    eos: int\n")
s = s.replace("[tk.eos_token_id]", "[self.eos]")
s = s.replace('print("train clips:",',
'''EOS = model.config.eos_token_id
if model.config.pad_token_id is None: model.config.pad_token_id = EOS
if model.config.decoder_start_token_id is None: model.config.decoder_start_token_id = model.config.bos_token_id
print("EOS:", EOS, "| BOS:", model.config.bos_token_id, "| decoder_start:", model.config.decoder_start_token_id, "| pad:", model.config.pad_token_id)
assert EOS is not None
print("train clips:",''')
s = s.replace("data_collator=Collator(proc)", "data_collator=Collator(proc, EOS)")
open("train.py", "w").write(s)
print("patched:", "self.eos" in s, "Collator(proc, EOS)" in s)

patched: True True


In [5]:
   !CUDA_VISIBLE_DEVICES=0 python train.py --steps 5000

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
cuda: True | capability: 7 | bf16: False | fp16: True
config.json: 100%|█████████████████████████████| 897/897 [00:00<00:00, 4.20MB/s]
tokenizer.json: 100%|███████████████████████| 1.99M/1.99M [00:00<00:00, 124MB/s]

model.safetensors: downloading bytes: █████████████▋        | 67.4MB, 4.35MB/s  
model.safetensors: reconstructing file:  61%|███▋  | 65.7MB /  108MB            
model.safetensors: downloading bytes: █████████████████     | 84.2MB, 7.64MB/s  
model.safetensors: downloading bytes: ██████████████████████|  103MB, 9.64MB/s  
model.safetensors: reconstructing file: 100%|██████|  108MB /  108MB, 10.3MB/s  
README.md: 100%|███████████████████████████████| 570/570 [00:00<00:00, 1.80MB/s]

data/train-00000-of-00012.parquet: downloading bytes: ██▌   |  239MB, 16.6MB/s  
data/train-00000-of-00012.parquet: downloading bytes: ████▌ |  436MB, 33.8MB/s  
data/